# Interview Prep: 20 - Advanced Deployment Strategies

Deploying code is easy; deploying code without breaking things is hard. This notebook covers the advanced release strategies used by high-engineering teams to ensure zero downtime and safe feature rollouts.

---

## 1. Blue-Green vs. Canary vs. A/B Testing

### **Interview Question**: "What is the main advantage of a Canary release over a Blue-Green deployment?"

**The Breakdown**:
- **Blue-Green**: You have two identical environments (Blue and Green). Traffic is swapped instantly via the load balancer. 
  - *Pro*: Instant rollback.
  - *Con*: Requires double the infrastructure costs.
- **Canary**: You roll out the new version to a small subset of users (e.g., 5%) first. If metrics look good, you roll out to the rest. 
  - *Pro*: Safest way to detect bugs in production with minimal impact.
- **A/B Testing**: A business experiment where users see different features to measure conversion/engagement. Not primarily a deployment risk-reduction strategy.

---

## 2. Zero-Downtime Database Migrations

### **Interview Question**: "How do you rename a database column in a high-traffic production system without downtime?"

**Answer**: Use the **Expand/Contract (Parallel Change)** pattern.
1. **Expand**: Add the new column (but keep the old one). Update the code to **write to both** columns but read from the old one.
2. **Migrate**: Run a background job to copy data from the old column to the new one.
3. **Switch**: Update the code to **read from the new** column.
4. **Contract**: Wait for all code pointers to be updated, then delete the old column.

---

## 3. Feature Flags

### **Interview Question**: "What is the difference between 'Deployment' and 'Release'?"

**Answer**:
- **Deployment**: The act of pushing code to servers/production.
- **Release**: The act of making that code functional for users. 
**Feature Flags** allow you to deploy code in a "dark" state. You can then release it to specific users at any time without a new deployment.

---

## 4. Rollback Strategies

### **Interview Question**: "If a deployment causes a spike in 500 errors, how do you decide between 'Rolling back' and 'Fixing forward'?"

**Answer**:
- **Rollback**: Default strategy for critical bugs. Speed to recovery is the #1 priority.
- **Fix Forward**: Only for minor bugs or when the deployment included a non-reversible operation (like a stateful DB migration that broke compatibility).

---

## 5. Coding Scenario: Conditional Rollouts

**Task**: Write a simple logic that determines if a user should see a new feature based on a hash of their `user_id` (ensures a consistent experience for that user).


In [ ]:
import hashlib

def should_show_feature(user_id, rollout_percentage):
    # Deterministic hash of the user_id
    hash_val = int(hashlib.md5(str(user_id).encode()).hexdigest(), 16)
    # Get a value between 0 and 99
    score = hash_val % 100
    return score < rollout_percentage

# Test for User 456 at 10% rollout
is_enabled = should_show_feature(456, 10)
print(f"Feature enabled for user 456: {is_enabled}")